In [3]:
#!/usr/bin/env python3
"""
File: deploy_minimal_unet.py

This script exports a minimal version of UNetSplit (a minimal segmentation CNN)
to ONNX, converts it into FINN‐ONNX format, applies the standard transforms and
then builds a Zynq project (for a PYNQ‐Z2 board) and generates a PYNQ driver.

We include a monkey‐patch for the MultiThreshold custom op so that it supports
a minimal get_folded_input_shape() method (returning a dummy shape) to satisfy
the downstream InsertIODMA transform.
"""

import os
import torch
import torch.nn as nn

# QONNX / FINN imports
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.core.datatype import DataType
from finn.util.basic import make_build_dir
from brevitas.export import export_qonnx

# Standard FINN transforms
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.lower_convs_to_matmul import LowerConvsToMatMul
# (Since this is an 8-bit network we skip ConvertBipolarMatMulToXnorPopcount)
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.general import (
    RemoveUnusedTensors,
    GiveUniqueNodeNames,
    GiveReadableTensorNames,
    RemoveStaticGraphInputs,
)
import finn.transformation.streamline.absorb as absorb
from finn.transformation.streamline import Streamline
from finn.transformation.move_reshape import RemoveCNVtoFCFlatten

# Dataflow transforms (old‐style)
import finn.transformation.fpgadataflow.convert_to_hw_layers as to_hw
from finn.transformation.fpgadataflow.specialize_layers import SpecializeLayers
from finn.transformation.fpgadataflow.make_zynq_proj import ZynqBuild
from finn.transformation.fpgadataflow.make_pynq_driver import MakePYNQDriver
from finn.transformation.fpgadataflow.set_exec_mode import SetExecMode

from finn.util.visualization import showInNetron

# Minimal model – our UNetSplit minimal variant (from your split_unet.py)
from split_unet import UNetSplit


# Create a unique build directory (e.g., /tmp/finn_dev_<...>)
build_dir = make_build_dir("unet_minimal_int_build_noInferConvWeights")
MODEL_WEIGHTS = "./best_unet_weights.pth"  # Path to your trained minimal model weights

# 1) Load your minimal model (here UNetSplit is our minimal segmentation CNN)
model_torch = UNetSplit(
    in_ch=1,
    out_ch=1,
    encoder_channels=[64,128,256,512],
    decoder_channels=[512,256,128,64],
    bottleneck_channels=1024,
    weight_bit_width=8,
    act_bit_width=8
)
state_dict = torch.load(MODEL_WEIGHTS, map_location="cpu")
model_torch.load_state_dict(state_dict)

# 2) Export to QONNX using a dummy input in NHWC format.
export_onnx_path = os.path.join(build_dir, "unet_minimal_export.onnx")
export_qonnx(model_torch, torch.randn(1, 1, 128, 128), export_onnx_path)
qonnx_cleanup(export_onnx_path, out_file=export_onnx_path)
model = ModelWrapper(export_onnx_path)
model = model.transform(ConvertQONNXtoFINN())
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(RemoveStaticGraphInputs())
model.save(build_dir + "/unet_minimal_tidy.onnx")
           
showInNetron(build_dir + "/unet_minimal_tidy.onnx")

Node 'Mul_2' has been removed and model saved as '/tmp/finn_dev_komaro/unet_minimal_int_build_noInferConvWeights8efsbnpx/unet_minimal_tidy_no_mul.onnx'
Stopping http://0.0.0.0:8081
Serving '/tmp/finn_dev_komaro/unet_minimal_int_build_noInferConvWeights8efsbnpx/unet_minimal_tidy_no_mul.onnx' at http://0.0.0.0:8081


In [4]:
from finn.transformation.streamline import Streamline
from qonnx.transformation.lower_convs_to_matmul import LowerConvsToMatMul
from qonnx.transformation.bipolar_to_xnor import ConvertBipolarMatMulToXnorPopcount
import finn.transformation.streamline.absorb as absorb
from finn.transformation.streamline.reorder import MakeMaxPoolNHWC, MoveScalarLinearPastInvariants
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.general import RemoveUnusedTensors

model = ModelWrapper(build_dir + "/unet_minimal_tidy.onnx")
model = model.transform(MoveScalarLinearPastInvariants())
model = model.transform(Streamline())
model = model.transform(LowerConvsToMatMul())
model = model.transform(MakeMaxPoolNHWC())
model = model.transform(absorb.AbsorbTransposeIntoMultiThreshold())
model = model.transform(ConvertBipolarMatMulToXnorPopcount())
model = model.transform(Streamline())
# absorb final add-mul nodes into TopK
model = model.transform(absorb.AbsorbScalarMulAddIntoTopK())
model = model.transform(InferDataLayouts())
model = model.transform(RemoveUnusedTensors())

model.save(build_dir + "/unet_minimal_streamlined.onnx")

showInNetron(build_dir + "/unet_minimal_streamlined.onnx")

Stopping http://0.0.0.0:8081
Serving '/tmp/finn_dev_komaro/unet_minimal_int_build_noInferConvWeights8efsbnpx/unet_minimal_streamlined.onnx' at http://0.0.0.0:8081


/home/komaro/デスクトップ/Cermak/finn/deps/qonnx/src/qonnx/transformation/infer_data_layouts.py:127: UserWarning: Assuming 4D input is NCHW
  warnings.warn("Assuming 4D input is NCHW")


In [6]:
from finn.util.basic import pynq_part_map
from finn.transformation.fpgadataflow.convert_to_hw_layers import *
from finn.transformation.fpgadataflow.create_dataflow_partition import CreateDataflowPartition
from finn.transformation.move_reshape import RemoveCNVtoFCFlatten
from finn.transformation.fpgadataflow.specialize_layers import SpecializeLayers
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.custom_op.registry import getCustomOp
import finn.transformation.streamline.absorb as absorb
from qonnx.core.modelwrapper import ModelWrapper

# Change this if you have a different PYNQ board
pynq_board = "Pynq-Z1"
fpga_part = pynq_part_map[pynq_board]
target_clk_ns = 10

# Load the model
model = ModelWrapper(build_dir + "/unet_minimal_streamlined.onnx")

# Apply transformations to convert all layers to hardware-compatible layers
model = model.transform(InferBinaryMatrixVectorActivation())  # https://finn.readthedocs.io/en/latest/source_code/finn.transformation.fpgadataflow.html
model = model.transform(InferQuantizedMatrixVectorActivation())  # https://finn.readthedocs.io/en/latest/source_code/finn.transformation.fpgadataflow.html
model = model.transform(InferLabelSelectLayer())  # https://finn.readthedocs.io/en/latest/source_code/finn.transformation.fpgadataflow.html
model = model.transform(InferThresholdingLayer())  # https://finn.readthedocs.io/en/latest/source_code/finn.transformation.fpgadataflow.html
model = model.transform(InferConvInpGen())  # https://finn.readthedocs.io/en/latest/source_code/finn.transformation.fpgadataflow.html
model = model.transform(InferStreamingMaxPool())  # https://finn.readthedocs.io/en/latest/source_code/finn.transformation.fpgadataflow.html
model = model.transform(RemoveCNVtoFCFlatten())  # https://finn.readthedocs.io/en/latest/source_code/finn.transformation.move_reshape.html
model = model.transform(absorb.AbsorbConsecutiveTransposes())  # https://finn.readthedocs.io/en/latest/source_code/finn.transformation.streamline.html
model = model.transform(InferDataLayouts())  # https://qonnx.readthedocs.io/en/latest/overview.html

# Convert MultiThreshold to hardware-compatible ThresholdingLayer
model = model.transform(InferThresholdingLayer())  # https://finn.readthedocs.io/en/latest/source_code/finn.transformation.fpgadataflow.html

# Finalize hardware transformation
model = model.transform(SpecializeLayers(fpga_part))  # https://finn.readthedocs.io/en/latest/nw_prep.html
model = model.transform(CreateDataflowPartition())  # https://finn.readthedocs.io/en/latest/source_code/finn.transformation.fpgadataflow.html

# Save the final hardware-compatible ONNX model
model.save(build_dir + "/unet_minimal_hw.onnx")

# Visualize the model in Netron
showInNetron(build_dir + "/unet_minimal_hw.onnx")

Stopping http://0.0.0.0:8081
Serving '/tmp/finn_dev_komaro/unet_minimal_int_build_noInferConvWeights8efsbnpx/unet_minimal_hw.onnx' at http://0.0.0.0:8081


In [17]:
from qonnx.core.modelwrapper import ModelWrapper

model = ModelWrapper(build_dir + "/unet_minimal_hw.onnx")

for node in model.graph.node:
    print(f"Node Name: {node.name}, Op Type: {node.op_type}")



Node Name: Transpose_0, Op Type: Transpose
Node Name: MatMul_0, Op Type: MatMul
Node Name: MultiThreshold_0, Op Type: MultiThreshold
Node Name: Transpose_1, Op Type: Transpose
Node Name: Mul_0, Op Type: Mul


In [12]:
parent_model = model.transform(CreateDataflowPartition())
parent_model.save(build_dir + "/unet_minimal_dataflow_parent.onnx")

showInNetron(build_dir + "/unet_minimal_dataflow_parent.onnx")

sdp_node = parent_model.get_nodes_by_op_type("StreamingDataflowPartition")[0]
sdp_node = getCustomOp(sdp_node)
dataflow_model_filename = sdp_node.get_nodeattr("model")
# save the dataflow partition with a different name for easier access
# and specialize the layers to HLS variants
dataflow_model = ModelWrapper(dataflow_model_filename)
dataflow_model = dataflow_model.transform(SpecializeLayers(fpga_part))
dataflow_model.save(build_dir + "/unet_minimal_dataflow_model.onnx")

Stopping http://0.0.0.0:8081
Serving '/tmp/finn_dev_komaro/unet_minimal_int_build_noInferConvWeightsfuul9yn1/unet_minimal_dataflow_parent.onnx' at http://0.0.0.0:8081


IndexError: list index out of range

In [4]:
from finn.transformation.fpgadataflow.make_zynq_proj import ZynqBuild
model = ModelWrapper(build_dir+"/unet_minimal_hw.onnx")
model = model.transform(ZynqBuild(platform = pynq_board, period_ns = target_clk_ns))

AttributeError: 'NoneType' object has no attribute 's'